# Lab 5 — OpenCV Functions in Python
**Programming for Artificial Intelligence**  
**Superior University Lahore**

This notebook covers all main OpenCV topics:
1. Reading & Writing Images
2. Displaying Images
3. Resizing & Cropping
4. Color Spaces
5. Drawing Shapes
6. Image Thresholding
7. Edge Detection
8. Contours & Shape Detection
9. Face Detection

## Setup — Import Libraries

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import os

# Helper function to display images inside notebook
def show(title, img, cmap=None):
    plt.figure(figsize=(7, 5))
    if cmap:
        plt.imshow(img, cmap=cmap)
    elif len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title, fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show_multiple(images, titles, cols=3):
    rows = (len(images) + cols - 1) // cols
    plt.figure(figsize=(6 * cols, 5 * rows))
    for i, (img, title) in enumerate(zip(images, titles)):
        plt.subplot(rows, cols, i + 1)
        if len(img.shape) == 2:
            plt.imshow(img, cmap='gray')
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(title, fontweight='bold')
        plt.axis('off')
    plt.tight_layout()
    plt.show()

print('✅ Libraries imported successfully!')
print(f'OpenCV version: {cv2.__version__}')

## Create a Sample Image (since we have no image file)
We'll create a synthetic image using NumPy to demonstrate all OpenCV functions.

In [ ]:
# Create a colorful sample image 400x600
sample = np.zeros((400, 600, 3), dtype=np.uint8)

# Add colored sections
sample[0:200, 0:200]   = [255, 100, 100]   # Blue-ish
sample[0:200, 200:400] = [100, 255, 100]   # Green-ish
sample[0:200, 400:600] = [100, 100, 255]   # Red-ish
sample[200:400, 0:300] = [200, 200, 50]    # Cyan-ish
sample[200:400, 300:600] = [200, 50, 200]  # Purple-ish

# Add some gradient noise
noise = np.random.randint(0, 40, sample.shape, dtype=np.uint8)
sample = cv2.add(sample, noise)

show('Sample Image (created with NumPy)', sample)
print(f'Image shape: {sample.shape}  →  Height={sample.shape[0]}, Width={sample.shape[1]}, Channels={sample.shape[2]}')

---
## 1. Reading & Writing Images
- `cv2.imread()` — reads an image from disk
- `cv2.imwrite()` — saves an image to disk

In [ ]:
# Save our sample image to disk
cv2.imwrite('sample.jpg', sample)
print('✅ Image saved as sample.jpg')

# Read image back from disk
img = cv2.imread('sample.jpg')
print(f'✅ Image read from disk')
print(f'   Shape  : {img.shape}')
print(f'   Dtype  : {img.dtype}')
print(f'   Min/Max pixel value: {img.min()} / {img.max()}')

show('Image read with cv2.imread()', img)

---
## 2. Displaying Images
In Jupyter notebooks we use `matplotlib` to display images instead of `cv2.imshow()` (which opens a separate window).

In [ ]:
# cv2.imshow() is for desktop apps — in notebooks we use matplotlib
# Convert BGR (OpenCV default) to RGB (matplotlib default)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img)          # Wrong colors (BGR shown as RGB)
axes[0].set_title('BGR (wrong colors)', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_rgb)      # Correct colors
axes[1].set_title('RGB (correct colors)', fontweight='bold')
axes[1].axis('off')

plt.suptitle('BGR vs RGB Display', fontsize=14)
plt.tight_layout()
plt.show()

print('Note: OpenCV reads images in BGR format, matplotlib expects RGB.')
print('Always use cv2.cvtColor(img, cv2.COLOR_BGR2RGB) before displaying!')

---
## 3. Resizing & Cropping
- `cv2.resize()` — resize image to specific dimensions
- Array slicing `[y1:y2, x1:x2]` — crop a region

In [ ]:
# ── Resizing ──
img_small  = cv2.resize(img, (300, 200))          # Fixed size
img_large  = cv2.resize(img, (900, 600))          # Larger
img_half   = cv2.resize(img, None, fx=0.5, fy=0.5)  # 50% scale

print('Original size :', img.shape[:2])
print('Resized small :', img_small.shape[:2])
print('Resized large :', img_large.shape[:2])
print('Half scale    :', img_half.shape[:2])

show_multiple(
    [img, img_small, img_half],
    ['Original (600x400)', 'Resized (300x200)', 'Half Scale (300x200)']
)

# ── Cropping ──
# Syntax: img[start_row : end_row, start_col : end_col]
crop_top_left     = img[0:200,   0:300]     # Top-left region
crop_center       = img[100:300, 150:450]   # Center region
crop_bottom_right = img[200:400, 300:600]   # Bottom-right region

show_multiple(
    [crop_top_left, crop_center, crop_bottom_right],
    ['Crop: Top-Left', 'Crop: Center', 'Crop: Bottom-Right']
)
print('✅ Resizing and Cropping done!')

---
## 4. Color Spaces
- `cv2.cvtColor()` — convert between color spaces
- Common: BGR → Gray, BGR → HSV, BGR → RGB

In [ ]:
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)   # Grayscale
img_hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)    # HSV
img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    # RGB
img_lab  = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)    # LAB

# Show all color spaces
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Color Space Conversions', fontsize=15, fontweight='bold')

axes[0,0].imshow(img_rgb);  axes[0,0].set_title('Original (RGB)');  axes[0,0].axis('off')
axes[0,1].imshow(img_gray, cmap='gray'); axes[0,1].set_title('Grayscale'); axes[0,1].axis('off')
axes[0,2].imshow(img_hsv);  axes[0,2].set_title('HSV');  axes[0,2].axis('off')
axes[1,0].imshow(img_lab);  axes[1,0].set_title('LAB');  axes[1,0].axis('off')

# Show individual HSV channels
h, s, v = cv2.split(img_hsv)
axes[1,1].imshow(h, cmap='hsv');  axes[1,1].set_title('H channel (Hue)');  axes[1,1].axis('off')
axes[1,2].imshow(v, cmap='gray'); axes[1,2].set_title('V channel (Value/Brightness)'); axes[1,2].axis('off')

plt.tight_layout()
plt.show()
print('✅ Color space conversions done!')

---
## 5. Drawing Shapes
- `cv2.line()` — draw a line
- `cv2.rectangle()` — draw a rectangle
- `cv2.circle()` — draw a circle
- `cv2.putText()` — add text

In [ ]:
# Create a blank white canvas
canvas = np.ones((500, 700, 3), dtype=np.uint8) * 255

# cv2.line(image, start_point, end_point, color_BGR, thickness)
cv2.line(canvas, (50, 50), (650, 50),   (0, 0, 255),   3)   # Red line
cv2.line(canvas, (50, 80), (650, 80),   (0, 255, 0),   3)   # Green line
cv2.line(canvas, (50, 110), (650, 110), (255, 0, 0),   3)   # Blue line
cv2.line(canvas, (100, 150), (600, 400),(128, 0, 128),  5)  # Diagonal purple

# cv2.rectangle(image, top_left, bottom_right, color, thickness)
cv2.rectangle(canvas, (50, 160),  (250, 320), (0, 128, 255), 3)    # Orange outline
cv2.rectangle(canvas, (280, 160), (480, 320), (0, 200, 100), -1)   # Filled green

# cv2.circle(image, center, radius, color, thickness)
cv2.circle(canvas, (580, 240), 70,  (255, 50, 50),  3)    # Blue outline circle
cv2.circle(canvas, (150, 420), 50,  (0, 0, 200),   -1)   # Filled red circle
cv2.circle(canvas, (350, 420), 50,  (0, 180, 0),   -1)   # Filled green circle
cv2.circle(canvas, (550, 420), 50,  (200, 0, 0),   -1)   # Filled blue circle

# cv2.putText(image, text, position, font, scale, color, thickness)
cv2.putText(canvas, 'OpenCV Drawing', (160, 490),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 3)

show('Drawing Shapes with OpenCV', canvas)
print('✅ Shapes drawn: lines, rectangles, circles, text!')

---
## 6. Image Thresholding
- `cv2.threshold()` — simple thresholding (converts to black & white)
- `cv2.adaptiveThreshold()` — works better for uneven lighting

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Simple Thresholding
# Pixels > 127 become white (255), others become black (0)
_, thresh_binary   = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
_, thresh_inv      = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
_, thresh_trunc    = cv2.threshold(gray, 127, 255, cv2.THRESH_TRUNC)
_, thresh_otsu     = cv2.threshold(gray, 0,   255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Adaptive Thresholding (adjusts threshold for different regions)
thresh_adaptive = cv2.adaptiveThreshold(
    gray, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 11, 2
)

show_multiple(
    [gray, thresh_binary, thresh_inv, thresh_trunc, thresh_otsu, thresh_adaptive],
    ['Grayscale', 'Binary (>127=white)', 'Binary Inverted',
     'Truncate', "Otsu's Method", 'Adaptive (Gaussian)'],
    cols=3
)
print('✅ Thresholding done!')

---
## 7. Edge Detection
- `cv2.Canny()` — finds edges in an image using gradient detection
- Also shows Sobel and Laplacian for comparison

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Blur first to reduce noise (makes edge detection cleaner)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

# Canny Edge Detection — cv2.Canny(image, low_threshold, high_threshold)
edges_low    = cv2.Canny(blurred, 30,  100)   # More edges (sensitive)
edges_mid    = cv2.Canny(blurred, 100, 200)   # Balanced
edges_high   = cv2.Canny(blurred, 150, 300)   # Fewer edges (strict)

# Sobel Edge Detection (detects edges in X and Y direction)
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
sobel_combined = cv2.magnitude(sobel_x, sobel_y).astype(np.uint8)

# Laplacian
laplacian = cv2.Laplacian(gray, cv2.CV_64F)
laplacian = np.uint8(np.absolute(laplacian))

show_multiple(
    [gray, blurred, edges_low, edges_mid, sobel_combined, laplacian],
    ['Grayscale', 'Gaussian Blur', 'Canny (low)', 'Canny (mid)', 'Sobel', 'Laplacian'],
    cols=3
)
print('✅ Edge detection done!')

---
## 8. Contours & Shape Detection
- `cv2.findContours()` — finds outlines of shapes
- `cv2.drawContours()` — draws those outlines
- We can also detect shape names using corner count

In [ ]:
# Create a canvas with clear shapes for contour detection
shape_img = np.zeros((500, 700, 3), dtype=np.uint8)
cv2.rectangle(shape_img, (50, 50),   (200, 180), (255, 255, 255), -1)  # Rectangle
cv2.circle(shape_img,    (350, 120), 80,          (255, 255, 255), -1)  # Circle
pts = np.array([[580,50],[650,180],[510,180]], np.int32)                # Triangle
cv2.fillPoly(shape_img, [pts], (255, 255, 255))
cv2.rectangle(shape_img, (50, 280),  (350, 450), (255, 255, 255), -1)  # Wide rect
cv2.circle(shape_img,    (500, 370), 90,          (255, 255, 255), -1)  # Big circle

# Convert to gray and find contours
gray_shapes = cv2.cvtColor(shape_img, cv2.COLOR_BGR2GRAY)
_, thresh   = cv2.threshold(gray_shapes, 127, 255, cv2.THRESH_BINARY)
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Draw contours and label shapes
result = shape_img.copy()
for cnt in contours:
    # Approximate the contour shape
    epsilon = 0.04 * cv2.arcLength(cnt, True)
    approx  = cv2.approxPolyDP(cnt, epsilon, True)
    corners = len(approx)

    # Determine shape name
    if corners == 3:
        shape_name = 'Triangle'
        color = (0, 255, 0)
    elif corners == 4:
        shape_name = 'Rectangle'
        color = (255, 100, 0)
    else:
        shape_name = 'Circle'
        color = (0, 100, 255)

    # Draw contour
    cv2.drawContours(result, [cnt], -1, color, 3)

    # Label shape
    M = cv2.moments(cnt)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
        cv2.putText(result, shape_name, (cx-40, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

print(f'Total contours found: {len(contours)}')
show_multiple(
    [shape_img, thresh, result],
    ['Original Shapes', 'Binary Threshold', f'Contours + Shape Labels ({len(contours)} found)']
)
print('✅ Contour detection done!')

---
## 9. Face Detection
- `cv2.CascadeClassifier()` — loads a pre-trained face detector
- `detectMultiScale()` — detects faces in an image

> **Note:** Replace `'your_photo.jpg'` with a real photo to detect actual faces.

In [ ]:
import cv2

# Load the pre-trained face detector (comes with OpenCV)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade  = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

print('✅ Haar Cascade classifiers loaded!')
print()
print('To detect faces on your own photo, use this code:')
print()
print('  img  = cv2.imread("your_photo.jpg")')
print('  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)')
print()
print('  faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)')
print()
print('  for (x, y, w, h) in faces:')
print('      cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)')
print('      roi_gray = gray[y:y+h, x:x+w]')
print('      eyes = eye_cascade.detectMultiScale(roi_gray)')
print('      for (ex, ey, ew, eh) in eyes:')
print('          cv2.rectangle(img, (x+ex, y+ey), (x+ex+ew, y+ey+eh), (255, 0, 0), 2)')
print()
print('  show("Face Detection Result", img)')

# Demo — create a synthetic face-like image
face_demo = np.ones((300, 300, 3), dtype=np.uint8) * 200
cv2.circle(face_demo, (150, 150), 120, (180, 160, 140), -1)  # Face
cv2.circle(face_demo, (110, 120), 18,  (50, 50, 50),   -1)  # Left eye
cv2.circle(face_demo, (190, 120), 18,  (50, 50, 50),   -1)  # Right eye
cv2.ellipse(face_demo, (150, 180), (40, 20), 0, 0, 180, (80, 60, 60), -1)  # Mouth
cv2.rectangle(face_demo, (60, 30), (240, 270), (0, 255, 0), 3)   # Face bounding box
cv2.rectangle(face_demo, (95, 105),(130, 138), (255, 0, 0), 2)   # Left eye box
cv2.rectangle(face_demo, (173, 105),(208, 138),(255, 0, 0), 2)   # Right eye box
cv2.putText(face_demo, 'Face', (80, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 0), 2)

show('Face Detection Demo (Simulated)', face_demo)

---
## Bonus — Image Blurring & Filters

In [ ]:
img_gaussian = cv2.GaussianBlur(img, (15, 15), 0)     # Gaussian blur
img_median   = cv2.medianBlur(img, 15)                 # Median blur
img_bilateral= cv2.bilateralFilter(img, 9, 75, 75)    # Bilateral (preserves edges)

show_multiple(
    [img, img_gaussian, img_median, img_bilateral],
    ['Original', 'Gaussian Blur', 'Median Blur', 'Bilateral Filter']
)
print('✅ Blurring filters done!')

---
## Summary

| Topic | Function Used |
|---|---|
| Reading Images | `cv2.imread()` |
| Writing Images | `cv2.imwrite()` |
| Displaying | `cv2.cvtColor()` + matplotlib |
| Resizing | `cv2.resize()` |
| Cropping | Array slicing `[y1:y2, x1:x2]` |
| Color Spaces | `cv2.cvtColor()` |
| Drawing | `cv2.line()`, `cv2.rectangle()`, `cv2.circle()`, `cv2.putText()` |
| Thresholding | `cv2.threshold()`, `cv2.adaptiveThreshold()` |
| Edge Detection | `cv2.Canny()`, Sobel, Laplacian |
| Contours | `cv2.findContours()`, `cv2.drawContours()` |
| Face Detection | `cv2.CascadeClassifier()` |